# 18 Randomized Reference-Drawing Baseline

## Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "18_randomized_drawing_baseline_weighted"
FIGURE_DIR = NOTEBOOK_OUTPUT_DIR / "figures"
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / "tables"
DATA_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "data"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENDER_CATEGORIES = ["MM", "MW", "WM", "WW"]

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Functions

In [2]:
def standardize_edges(edges):
    out = edges[["citing_paper_id", "cited_paper_id"]].copy()
    out["citing_paper_id"] = out["citing_paper_id"].astype(str)
    out["cited_paper_id"] = out["cited_paper_id"].astype(str)
    return out

def load_final_core():
    papers = pd.read_pickle(PROCESSED_DIR / "papers_final.pkl")
    cs_papers = pd.read_pickle(PROCESSED_DIR / "cs_papers_final.pkl")
    all_edges = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_final.pkl"))
    all_to_cs = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_all_to_cs_final.pkl"))
    cs_to_cs = standardize_edges(pd.read_pickle(PROCESSED_DIR / "citation_edges_cs_to_cs_final.pkl"))
    papers["id"] = papers["id"].astype(str)
    cs_papers["id"] = cs_papers["id"].astype(str)
    return papers, cs_papers, all_edges, all_to_cs, cs_to_cs

def known_cs(cs_papers):
    return cs_papers[cs_papers["first_last_cat"].isin(GENDER_CATEGORIES)].copy()

def aggregate_oe(observed, expected, baseline, citation_slice):
    rows = []
    for g in GENDER_CATEGORIES:
        o = float(observed.get(g, 0.0))
        e = float(expected.get(g, 0.0))
        oe = o / e if e > 0 else np.nan
        rows.append({
            "gender_cat": g,
            "observed_citations": o,
            "expected_citations": e,
            "oe_ratio": oe,
            "pct_diff_from_expected": (oe - 1.0) * 100 if pd.notna(oe) else np.nan,
            "baseline": baseline,
            "citation_slice": citation_slice,
        })
    return pd.DataFrame(rows)

def drawing_probability_table(citing_years, hist_year, paper_year):
    rows = []
    for y in sorted(set(map(int, citing_years))):
        prior_hist = hist_year[hist_year["citing_year"].lt(y)]
        for rule in ["all_prior", "age_0_10"]:
            if rule == "all_prior":
                pc = paper_year[paper_year["year"].lt(y)]
                hc = prior_hist[prior_hist["target_year"].lt(y)]
            else:
                pc = paper_year[paper_year["year"].between(y - 10, y)]
                hc = prior_hist[prior_hist["target_year"].between(y - 10, y)]

            pcount = pc.groupby("first_last_cat")["papers"].sum().reindex(GENDER_CATEGORIES, fill_value=0)
            hcount = hc.groupby("gender_cat")["citations"].sum().reindex(GENDER_CATEGORIES, fill_value=0)

            total_p = pcount.sum()
            weights = pcount + hcount
            total_w = weights.sum()

            for scheme in ["uniform", "prior_weighted"]:
                row = {
                    "citing_year": y,
                    "eligibility_rule": rule,
                    "drawing_scheme": scheme,
                }

                if scheme == "uniform" and total_p:
                    vals = pcount / total_p
                elif total_w:
                    vals = weights / total_w
                else:
                    vals = pd.Series(0, index=GENDER_CATEGORIES)

                for g in GENDER_CATEGORIES:
                    row[g] = float(vals[g])

                row["eligible_papers"] = int(total_p)
                rows.append(row)

    return pd.DataFrame(rows)

def prepare_observed(edges, year_map, known_meta):
    x = edges.copy()
    x["citing_year"] = x["citing_paper_id"].map(year_map)
    x = x.dropna(subset=["citing_year"]).copy()
    x["citing_year"] = x["citing_year"].astype(int)
    x = x.merge(
        known_meta[["cited_paper_id", "gender_cat"]],
        on="cited_paper_id",
        how="inner",
    )
    return x

def evaluate_draw(x, citation_slice, prob):
    pieces = []
    for (rule, scheme), p in prob.groupby(["eligibility_rule", "drawing_scheme"]):
        p = p[p["eligible_papers"].gt(0)]
        z = x.merge(p, on="citing_year", how="inner")
        obs = z["gender_cat"].value_counts().reindex(GENDER_CATEGORIES, fill_value=0)
        exp = z[GENDER_CATEGORIES].sum().reindex(GENDER_CATEGORIES, fill_value=0)
        out = aggregate_oe(
            obs,
            exp,
            f"random_draw_{rule}_{scheme}",
            citation_slice,
        )
        out["eligibility_rule"] = rule
        out["drawing_scheme"] = scheme
        out["n_edges"] = len(z)
        pieces.append(out)
    return pd.concat(pieces, ignore_index=True)

## Compute Randomized Expectations

In [3]:
papers, cs_papers, citation_edges_final, edges_all_to_cs, edges_cs_to_cs = load_final_core()

cs_known = known_cs(cs_papers).dropna(subset=["year"]).copy()
cs_known["year"] = cs_known["year"].astype(int)

year_map = papers.set_index("id")["year"]

known_meta = cs_known[["id", "year", "first_last_cat"]].rename(
    columns={
        "id": "cited_paper_id",
        "year": "target_year",
        "first_last_cat": "gender_cat",
    }
)

hist = citation_edges_final.copy()
hist["citing_year"] = hist["citing_paper_id"].map(year_map)
hist = hist.dropna(subset=["citing_year"]).copy()
hist["citing_year"] = hist["citing_year"].astype(int)
hist = hist.merge(known_meta, on="cited_paper_id", how="inner")

hist_year = (
    hist.groupby(["citing_year", "target_year", "gender_cat"])
    .size()
    .rename("citations")
    .reset_index()
)

paper_year = (
    cs_known.groupby(["year", "first_last_cat"])
    .size()
    .rename("papers")
    .reset_index()
)

all_obs = prepare_observed(edges_all_to_cs, year_map, known_meta)
cs_obs = prepare_observed(edges_cs_to_cs, year_map, known_meta)

prob = drawing_probability_table(
    pd.concat([all_obs["citing_year"], cs_obs["citing_year"]]).unique(),
    hist_year,
    paper_year,
)

draw_results = pd.concat([
    evaluate_draw(all_obs, "All→CS", prob),
    evaluate_draw(cs_obs, "CS→CS", prob),
], ignore_index=True)

display(draw_results)

,gender_cat,observed_citations,expected_citations,oe_ratio,pct_diff_from_expected,baseline,citation_slice,eligibility_rule,drawing_scheme,n_edges
0,MM,"6,942,653.0000","6,732,932.2867",1.0311,3.1148,random_draw_age_0_10_prior_weighted,All→CS,age_0_10,prior_weighted,8686237
1,MW,"641,435.0000","715,229.5340",0.8968,-10.3176,random_draw_age_0_10_prior_weighted,All→CS,age_0_10,prior_weighted,8686237
2,WM,"812,529.0000","911,290.1431",0.8916,-10.8375,random_draw_age_0_10_prior_weighted,All→CS,age_0_10,prior_weighted,8686237
3,WW,"289,620.0000","326,785.0363",0.8863,-11.3729,random_draw_age_0_10_prior_weighted,All→CS,age_0_10,prior_weighted,8686237
4,MM,"6,942,653.0000","6,465,236.5487",1.0738,7.3844,random_draw_age_0_10_uniform,All→CS,age_0_10,uniform,8686237
5,MW,"641,435.0000","754,325.3862",0.8503,-14.9657,random_draw_age_0_10_uniform,All→CS,age_0_10,uniform,8686237
6,WM,"812,529.0000","1,041,432.4840",0.7802,-21.9797,random_draw_age_0_10_uniform,All→CS,age_0_10,uniform,8686237
7,WW,"289,620.0000","425,242.5812",0.6811,-31.8930,random_draw_age_0_10_uniform,All→CS,age_0_10,uniform,8686237
8,MM,"6,942,653.0000","7,183,908.9800",0.9664,-3.3583,random_draw_all_prior_prior_weighted,All→CS,all_prior,prior_weighted,8686237
9,MW,"641,435.0000","549,294.0367",1.1677,16.7744,random_draw_all_prior_prior_weighted,All→CS,all_prior,prior_weighted,8686237


## Save Results

In [4]:
draw_results.to_csv(
    TABLE_DIR / "randomized_reference_drawing_oe_by_gender.csv",
    index=False,
)
prob.to_csv(
    TABLE_DIR / "randomized_reference_drawing_probability_table.csv",
    index=False,
)